In [1]:
import cv2
import mediapipe as mp
import csv
import os

# --- 1. Setup MediaPipe ---
mp_face_mesh = mp.solutions.face_mesh
face_mesh = mp_face_mesh.FaceMesh(refine_landmarks=True)

# --- 2. File Management ---
FILE_NAME = 'gaze_data.csv'
file_exists = os.path.isfile(FILE_NAME)

# --- 3. Initialize Counters ---
counts = {"center": 0, "left": 0, "right": 0}
if file_exists:
    with open(FILE_NAME, 'r') as f:
        reader = csv.DictReader(f)
        for row in reader:
            label = row.get('label')
            if label in counts:
                counts[label] += 1

cap = cv2.VideoCapture(0)

print(f"Current Cumulative Data Rows: {counts}")
print("CONTROLS: 'c'=Center, 'l'=Left, 'r'=Right, 'q'=Quit")

# Open in 'a' (append) mode to safely accumulate data rows
with open(FILE_NAME, mode='a', newline='') as f:
    writer = csv.writer(f)
    
    # Write header ONLY if the file is brand new (Added 'face_symmetry')
    if not file_exists:
        writer.writerow(['iris_pos', 'eye_width', 'nose_relative_pos', 'face_symmetry', 'label'])

    while cap.isOpened():
        success, frame = cap.read()
        if not success: break
        
        frame = cv2.flip(frame, 1)
        h, w, _ = frame.shape
        results = face_mesh.process(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))

        current_features = None

        if results.multi_face_landmarks:
            for face_landmarks in results.multi_face_landmarks:
                # --- Get Landmarks ---
                left_iris = face_landmarks.landmark[468]
                left_outer = face_landmarks.landmark[33]
                left_inner = face_landmarks.landmark[133]
                right_outer = face_landmarks.landmark[263] # Added for symmetry
                nose = face_landmarks.landmark[1]        

                left_edge = face_landmarks.landmark[234]  
                right_edge = face_landmarks.landmark[454] 

                # --- Calculations ---
                # 1. Eye width and iris position
                eye_width = left_inner.x - left_outer.x
                iris_pos = left_iris.x - left_outer.x
                
                # 2. Nose relative position
                face_width = right_edge.x - left_edge.x
                nose_relative_pos = (nose.x - left_edge.x) / face_width
                
                # 3. Face Symmetry (3D perspective rotation tracking)
                dist_to_left = abs(left_outer.x - nose.x)
                dist_to_right = abs(right_outer.x - nose.x)
                face_symmetry = dist_to_left / (dist_to_right + 1e-6)

                # --- Construct Upgraded Feature Array ---
                current_features = [iris_pos, eye_width, nose_relative_pos, face_symmetry]

                # Draw tracking visuals
                ix, iy = int(left_iris.x * w), int(left_iris.y * h)
                cv2.circle(frame, (ix, iy), 3, (0, 255, 0), -1)
                nx, ny = int(nose.x * w), int(nose.y * h)
                cv2.circle(frame, (nx, ny), 2, (255, 0, 0), -1)

        # --- UI Overlay ---
        frame[0:60, 0:w] = [0, 0, 0]
        status_text = f"C: {counts['center']} | L: {counts['left']} | R: {counts['right']}"
        cv2.putText(frame, status_text, (20, 40), cv2.FONT_HERSHEY_SIMPLEX, 0.8, (255, 255, 255), 2)

        # --- Key Handling ---
        key = cv2.waitKey(1) & 0xFF
        
        if current_features:
            if key == ord('c'):
                writer.writerow(current_features + ['center'])
                counts['center'] += 1
            elif key == ord('l'):
                writer.writerow(current_features + ['left'])
                counts['left'] += 1
            elif key == ord('r'):
                writer.writerow(current_features + ['right'])
                counts['right'] += 1

        cv2.imshow('Study Guardian: Data Collector', frame)
        if key == ord('q'): break

cap.release()
cv2.destroyAllWindows()
print(f"Finished! Total cumulative samples inside {FILE_NAME}: {counts}")

Current Cumulative Data Rows: {'center': 650, 'left': 650, 'right': 650}
CONTROLS: 'c'=Center, 'l'=Left, 'r'=Right, 'q'=Quit
Finished! Total cumulative samples inside gaze_data.csv: {'center': 650, 'left': 650, 'right': 650}
